In [1]:

from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Đọc dữ liệu lịch sử
history = pd.read_parquet(
    ROOT / "data/processed/features.parquet"
)

print("History shape:", history.shape)
print("Date range:", history["date"].min(), history["date"].max())


History shape: (3000888, 19)
Date range: 2013-01-01 00:00:00 2017-08-15 00:00:00


In [2]:

import inspect

from src.train_model import (
    prepare_data,
    build_origin_examples
)

print("=== PREPARE DATA ===")
print(inspect.getsource(prepare_data))

print("\n=== BUILD ORIGIN EXAMPLES ===")
print(inspect.getsource(build_origin_examples))


=== PREPARE DATA ===
def prepare_data(data: pd.DataFrame) -> pd.DataFrame:
    """Prepare compact typed data and one stable code per product family."""
    missing = set(SOURCE_COLUMNS).difference(data.columns)
    if missing:
        raise ValueError(f"Missing feature columns: {sorted(missing)}")
    df = data[SOURCE_COLUMNS].copy()
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    if df[["date", "store_nbr", "family", "sales"]].isna().any().any():
        raise ValueError("Missing dates, series keys, or targets")
    if df.duplicated(["date", "store_nbr", "family"]).any():
        raise ValueError("Duplicate date/store/family keys")
    if (df["sales"] < 0).any():
        raise ValueError("Negative unit sales")
    # Categories are derived from labels only: never from future sales values.
    df["family"] = df["family"].astype("category")
    df["family_code"] = df["family"].cat.codes.astype("int16")
    return df.sort_values("date").reset_index(drop=True)


=== BUILD OR

In [3]:

# Sử dụng preprocessing chính thức của Training
prepared = prepare_data(history)

# Chọn một cutoff trong giai đoạn Validation
cutoff = pd.Timestamp("2017-07-04")
horizon = 7

# Tạo features bằng logic Training
training_examples = build_origin_examples(
    prepared,
    cutoff,
    horizon
)

print("Training examples:", training_examples.shape)
print("Available columns:")
print(training_examples.columns.tolist())


Training examples: (12474, 17)
Available columns:
['date', 'store_nbr', 'family_code', 'sales', 'horizon_step', 'reference_date', 'baseline_reference', 'cutoff_sales', 'cutoff_lag_7', 'cutoff_lag_28', 'cutoff_rolling_mean_7', 'year', 'month', 'day_of_month', 'day_of_week', 'is_weekend', 'cutoff']


In [4]:

columns_to_check = [
    "cutoff",
    "date",
    "store_nbr",
    "family_code",
    "horizon_step",
    "baseline_reference",
    "cutoff_sales",
    "cutoff_lag_7",
    "cutoff_lag_28",
    "cutoff_rolling_mean_7"
]

display(
    training_examples[
        columns_to_check
    ].head(10)
)


,cutoff,date,store_nbr,family_code,horizon_step,baseline_reference,cutoff_sales,cutoff_lag_7,cutoff_lag_28,cutoff_rolling_mean_7
0,2017-07-04,2017-07-05,36,29,1,41.505001,44.723999,35.991001,69.931000,51.112000
1,2017-07-04,2017-07-05,37,4,1,0.000000,0.000000,0.000000,0.000000,0.000000
2,2017-07-04,2017-07-05,37,3,1,2961.000000,2479.000000,2438.000000,3441.000000,3032.571533
3,2017-07-04,2017-07-05,37,2,1,6.000000,11.000000,3.000000,6.000000,5.000000
4,2017-07-04,2017-07-05,37,1,1,0.000000,0.000000,0.000000,0.000000,0.000000
5,2017-07-04,2017-07-05,37,0,1,6.000000,3.000000,7.000000,18.000000,5.285714
6,2017-07-04,2017-07-05,36,32,1,1.000000,2.000000,0.000000,3.000000,0.571429
7,2017-07-04,2017-07-05,37,5,1,640.482971,588.768982,470.104004,534.749023,592.812988
8,2017-07-04,2017-07-05,36,31,1,0.000000,1.000000,1.000000,0.000000,0.285714
9,2017-07-04,2017-07-05,36,30,1,1035.027954,2794.001953,2370.272949,2816.672119,1415.758423


In [5]:

# Áp dụng preprocessing của Training
prepared = prepare_data(history)

# Lấy mapping thực tế
mapping_df = (
    prepared[["family", "family_code"]]
    .drop_duplicates()
    .copy()
)

# Chuyển sang dictionary
family_mapping = dict(
    zip(
        mapping_df["family"].astype(str),
        mapping_df["family_code"].astype(int)
    )
)

# Kiểm tra tính nhất quán
assert mapping_df["family"].is_unique
assert mapping_df["family_code"].is_unique
assert len(family_mapping) == history["family"].nunique()

print("Total families:", len(family_mapping))
print("BEVERAGES code:", family_mapping["BEVERAGES"])

display(mapping_df.sort_values("family"))


Total families: 33
BEVERAGES code: 3


,family,family_code
0,AUTOMOTIVE,0
6,BABY CARE,1
5,BEAUTY,2
4,BEVERAGES,3
3,BOOKS,4
2,BREAD/BAKERY,5
1,CELEBRATION,6
24,CLEANING,7
25,DAIRY,8
26,DELI,9


In [6]:

from src.forecast_features import build_future_calendar

cutoff = pd.Timestamp("2017-07-04")
store_nbr = 1
family = "BEVERAGES"
horizon = 7

# Features được tạo bởi Training
training_examples = build_origin_examples(
    prepared,
    cutoff,
    horizon
)

# Features được tạo bởi Inference
future = build_future_calendar(
    cutoff=cutoff,
    store_nbr=store_nbr,
    family=family,
    horizon=horizon
)

future["family_code"] = family_mapping[family]

# Chọn đúng chuỗi sản phẩm từ Training
training_sample = training_examples.loc[
    (training_examples["store_nbr"] == store_nbr)
    & (
        training_examples["family_code"]
        == family_mapping[family]
    )
].copy()

print("Training rows:", len(training_sample))
print("Inference rows:", len(future))


Training rows: 7
Inference rows: 7


In [7]:

calendar_features = [
    "year",
    "month",
    "day_of_month",
    "day_of_week",
    "is_weekend",
    "horizon_step",
    "family_code"
]

# Ghép theo ngày dự báo
comparison = future.merge(
    training_sample[["date"] + calendar_features],
    on="date",
    how="inner",
    suffixes=("_inference", "_training"),
    validate="one_to_one"
)

# Kiểm tra số dòng
assert len(comparison) == horizon

# So sánh các features
for feature in calendar_features:

    inference_values = comparison[
        f"{feature}_inference"
    ]

    training_values = comparison[
        f"{feature}_training"
    ]

    matches = inference_values.equals(training_values)

    # So sánh giá trị, không phụ thuộc dtype
    matches = (
        inference_values.to_numpy()
        == training_values.to_numpy()
    ).all()

    print(f"{feature}: {'PASS' if matches else 'FAIL'}")

    assert matches, f"Feature mismatch: {feature}"

print("\nCalendar feature parity: PASSED")


year: PASS
month: PASS
day_of_month: PASS
day_of_week: PASS
is_weekend: PASS
horizon_step: PASS
family_code: PASS

Calendar feature parity: PASSED


In [8]:

import inspect

source = inspect.getsource(
    build_origin_examples
).splitlines()

keywords = [
    "baseline_reference",
    "cutoff_sales",
    "cutoff_lag_7",
    "cutoff_lag_28",
    "cutoff_rolling_mean_7"
]

selected_lines = set()

for i, line in enumerate(source):
    if any(keyword in line for keyword in keywords):
        start = max(0, i - 3)
        end = min(len(source), i + 4)

        selected_lines.update(range(start, end))

for i in sorted(selected_lines):
    print(f"{i + 1:03d} | {source[i]}")


028 |     past = df.loc[
029 |         (df["date"] <= cutoff) & (df["date"].isin(reference_dates)),
030 |         ["date", *KEYS, "sales"],
031 |     ].rename(columns={"date": "reference_date", "sales": "baseline_reference"})
032 |     targets = targets.merge(past, on=["reference_date", *KEYS], how="left", validate="many_to_one")
033 | 
034 |     # Snapshot at the origin. All lag columns in this snapshot refer to earlier dates.
036 |         df["date"] == cutoff,
037 |         [*KEYS, "sales", "sales_lag_7", "sales_lag_28", "sales_rolling_mean_7"],
038 |     ].rename(columns={
039 |         "sales": "cutoff_sales",
040 |         "sales_lag_7": "cutoff_lag_7",
041 |         "sales_lag_28": "cutoff_lag_28",
042 |         "sales_rolling_mean_7": "cutoff_rolling_mean_7",
043 |     })
044 |     if snapshot.empty:
045 |         raise ValueError(f"No historical snapshot at cutoff={cutoff.date()}")


In [13]:

import inspect

source = inspect.getsource(
    build_origin_examples
).splitlines()

selected = set()

for i, line in enumerate(source):
    if (
        "reference_date" in line
        or "reference_dates" in line
    ):
        selected.update(
            range(
                max(0, i - 4),
                min(len(source), i + 4)
            )
        )

for i in sorted(selected):
    print(f"{i + 1:03d} | {source[i]}")


020 |     targets["horizon_step"] = (targets["date"] - cutoff).dt.days.astype("int16")
021 |     steps = targets["horizon_step"].to_numpy()
022 |     # For each future weekday, take its last same-weekday occurrence <= cutoff.
023 |     cycles = ((steps - 1) // 7) + 1
024 |     targets["reference_date"] = targets["date"] - pd.to_timedelta(cycles * 7, unit="D")
025 | 
026 |     # Fast calendar-specific lookups, from historical data only.
027 |     reference_dates = targets["reference_date"].unique()
028 |     past = df.loc[
029 |         (df["date"] <= cutoff) & (df["date"].isin(reference_dates)),
030 |         ["date", *KEYS, "sales"],
031 |     ].rename(columns={"date": "reference_date", "sales": "baseline_reference"})
032 |     targets = targets.merge(past, on=["reference_date", *KEYS], how="left", validate="many_to_one")
033 | 
034 |     # Snapshot at the origin. All lag columns in this snapshot refer to earlier dates.
035 |     snapshot = df.loc[


In [14]:

import inspect

source = inspect.getsource(
    build_origin_examples
).splitlines()

# Chỉ tìm phần tạo reference_date
for i, line in enumerate(source):
    if "reference_date" in line:
        start = max(0, i - 3)
        end = min(len(source), i + 4)

        print(f"\n--- Lines {start + 1}-{end} ---")

        for j in range(start, end):
            print(f"{j + 1:03d} | {source[j]}")



--- Lines 21-27 ---
021 |     steps = targets["horizon_step"].to_numpy()
022 |     # For each future weekday, take its last same-weekday occurrence <= cutoff.
023 |     cycles = ((steps - 1) // 7) + 1
024 |     targets["reference_date"] = targets["date"] - pd.to_timedelta(cycles * 7, unit="D")
025 | 
026 |     # Fast calendar-specific lookups, from historical data only.
027 |     reference_dates = targets["reference_date"].unique()

--- Lines 24-30 ---
024 |     targets["reference_date"] = targets["date"] - pd.to_timedelta(cycles * 7, unit="D")
025 | 
026 |     # Fast calendar-specific lookups, from historical data only.
027 |     reference_dates = targets["reference_date"].unique()
028 |     past = df.loc[
029 |         (df["date"] <= cutoff) & (df["date"].isin(reference_dates)),
030 |         ["date", *KEYS, "sales"],

--- Lines 26-32 ---
026 |     # Fast calendar-specific lookups, from historical data only.
027 |     reference_dates = targets["reference_date"].unique()
028 |     pa

In [15]:

import importlib
import pandas as pd
import src.forecast_features as ff

# Nạp lại module sau khi chỉnh sửa file Python
importlib.reload(ff)

cutoff = pd.Timestamp("2017-07-04")
store_nbr = 1
family = "BEVERAGES"

for horizon in [7, 10]:

    # Reference dates theo Training
    training = build_origin_examples(
        prepared, cutoff, horizon
    )

    training = training.loc[
        (training["store_nbr"] == store_nbr)
        & (training["family_code"] == family_mapping[family]),
        ["date", "reference_date"]
    ].sort_values("date").reset_index(drop=True)

    # Reference dates theo Inference
    inference = ff.build_future_calendar(
        cutoff=cutoff,
        store_nbr=store_nbr,
        family=family,
        horizon=horizon
    )

    inference = inference[
        ["date", "reference_date"]
    ].sort_values("date").reset_index(drop=True)

    # So sánh hai cách tạo reference_date
    pd.testing.assert_frame_equal(
        training,
        inference,
        check_dtype=False
    )

    print(f"Horizon {horizon}: PASS")

print("\nReference date parity: PASSED")


Horizon 7: PASS
Horizon 10: PASS

Reference date parity: PASSED


In [16]:

import importlib
import numpy as np
import pandas as pd
import src.forecast_features as ff

importlib.reload(ff)

cutoff = pd.Timestamp("2017-07-04")
store_nbr = 1
family = "BEVERAGES"

features_to_compare = [
    "baseline_reference",
    "cutoff_sales",
    "cutoff_lag_7",
    "cutoff_lag_28",
    "cutoff_rolling_mean_7"
]

for horizon in [7, 10]:

    # Features từ Training
    training = build_origin_examples(
        prepared, cutoff, horizon
    )

    training = training.loc[
        (training["store_nbr"] == store_nbr)
        & (training["family_code"] == family_mapping[family])
    ].sort_values("date").reset_index(drop=True)

    # Features từ Inference
    future = ff.build_future_calendar(
        cutoff, store_nbr, family, horizon
    )

    future["family_code"] = family_mapping[family]

    inference = ff.attach_historical_features(
        future, prepared, cutoff
    ).sort_values("date").reset_index(drop=True)

    assert len(training) == len(inference) == horizon

    # Kiểm tra đúng ngày dự báo
    pd.testing.assert_series_equal(
        training["date"],
        inference["date"],
        check_names=False
    )

    # So sánh từng historical feature
    for feature in features_to_compare:
        np.testing.assert_allclose(
            training[feature].to_numpy(dtype=float),
            inference[feature].to_numpy(dtype=float),
            rtol=0,
            atol=1e-5,
            equal_nan=True
        )

    print(f"Horizon {horizon}: PASS")

print("\nHistorical feature parity: PASSED")


Horizon 7: PASS
Horizon 10: PASS

Historical feature parity: PASSED


In [17]:

import mlflow
import mlflow.sklearn

from src.forecast_pipeline import forecast_sales

# Kết nối MLflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")

# Tải mô hình Gradient Boosting đã đăng ký
model = mlflow.sklearn.load_model(
    "models:/sales_forecasting_gradient_boosting@candidate"
)

# Dự báo thực tế
forecast = forecast_sales(
    model=model,
    history=prepared,
    family_mapping=family_mapping,
    cutoff="2017-08-15",
    store_nbr=1,
    family="BEVERAGES",
    horizon=7,
)

display(forecast)


,date,store_nbr,family,horizon_step,predicted_sales
0,2017-08-16,1,BEVERAGES,1,2093.065656
1,2017-08-17,1,BEVERAGES,2,2172.296788
2,2017-08-18,1,BEVERAGES,3,1235.694209
3,2017-08-19,1,BEVERAGES,4,1743.100802
4,2017-08-20,1,BEVERAGES,5,1070.913450
5,2017-08-21,1,BEVERAGES,6,2030.570979
6,2017-08-22,1,BEVERAGES,7,1874.197387


In [18]:

import numpy as np

from src.inference import predict_sales

cutoff = pd.Timestamp("2017-07-04")
horizon = 7

# Cách cũ: tạo examples bằng Training
training_examples = build_origin_examples(
    prepared, cutoff, horizon
)

training_sample = training_examples.loc[
    (training_examples["store_nbr"] == 1)
    & (
        training_examples["family_code"]
        == family_mapping["BEVERAGES"]
    )
].sort_values("date").reset_index(drop=True)

old_predictions = predict_sales(
    model, training_sample
)

# Cách mới: sử dụng Forecasting Pipeline
new_forecast = forecast_sales(
    model=model,
    history=prepared,
    family_mapping=family_mapping,
    cutoff=cutoff,
    store_nbr=1,
    family="BEVERAGES",
    horizon=horizon,
)

# Đối chiếu ngày dự báo
np.testing.assert_array_equal(
    training_sample["date"].to_numpy(),
    new_forecast["date"].to_numpy()
)

# Đối chiếu doanh số dự báo
np.testing.assert_allclose(
    old_predictions.to_numpy(),
    new_forecast["predicted_sales"].to_numpy(),
    rtol=0,
    atol=1e-5,
)

print("End-to-end forecast parity: PASSED")


End-to-end forecast parity: PASSED


In [19]:

import json
import pandas as pd

from src.train_model import prepare_data

mapping_path = ROOT / "data/processed/family_mapping.json"

# Mapping đang được API sử dụng
saved_mapping = json.loads(
    mapping_path.read_text(encoding="utf-8")
)

# Mapping được tạo từ toàn bộ dữ liệu
raw_history = pd.read_parquet(
    ROOT / "data/processed/features.parquet"
)

prepared = prepare_data(raw_history)

pairs = prepared[
    ["family", "family_code"]
].drop_duplicates()

actual_mapping = {
    str(family): int(code)
    for family, code in zip(
        pairs["family"],
        pairs["family_code"]
    )
}

print("Saved families:", len(saved_mapping))
print("Actual families:", len(actual_mapping))

# Tìm các family có mã không khớp
differences = {
    family: {
        "saved": saved_mapping.get(family),
        "actual": actual_code
    }
    for family, actual_code in actual_mapping.items()
    if saved_mapping.get(family) != actual_code
}

print("Mismatched families:", len(differences))
print("First differences:")

for family, values in list(differences.items())[:10]:
    print(family, values)


Saved families: 33
Actual families: 33
Mismatched families: 30
First differences:
CELEBRATION {'saved': 25, 'actual': 6}
BREAD/BAKERY {'saved': 26, 'actual': 5}
BOOKS {'saved': 27, 'actual': 4}
BEVERAGES {'saved': 28, 'actual': 3}
BEAUTY {'saved': 29, 'actual': 2}
BABY CARE {'saved': 30, 'actual': 1}
SCHOOL AND OFFICE SUPPLIES {'saved': 15, 'actual': 31}
PRODUCE {'saved': 1, 'actual': 30}
PREPARED FOODS {'saved': 2, 'actual': 29}
POULTRY {'saved': 3, 'actual': 28}


In [20]:

# Kiểm tra số nhóm sản phẩm
assert len(actual_mapping) == 33

# Lưu lại mapping đúng với preprocessing
mapping_path.write_text(
    json.dumps(
        actual_mapping,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

# Kiểm tra lại sau khi lưu
loaded_mapping = json.loads(
    mapping_path.read_text(encoding="utf-8")
)

assert loaded_mapping == actual_mapping

print("Family mapping synchronized: PASSED")
print("BEVERAGES code:", loaded_mapping["BEVERAGES"])


Family mapping synchronized: PASSED
BEVERAGES code: 3
